# Data Exploration

## Purpose
Transform the raw dataset into a cleaned, feature-engineered dataset aligned for training and future Vertex AI deployment.

## Imports & Configuration

In [ ]:
# Standard libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

## Load Project Secrets (Colab / userdata)

In [ ]:
from google.colab import userdata

PROJECT_ID = userdata.get("GCP_PROJECT_ID")
GITHUB_USER = userdata.get("GITHUB_USER")
GCS_BUCKET = userdata.get("GCS_BUCKET")
TRAINING_PREFIX = userdata.get("TRAINING_PREFIX")
DATA_PREP_PREFIX = userdata.get("DATA_PREP_PREFIX")
REPO_NAME = userdata.get("REPO_NAME")
REGION = userdata.get("REGION")

REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
CLONE_PATH = f"/content/{REPO_NAME}"

## Authenticate & Configure gcloud

In [ ]:
!gcloud auth login --quiet
!gcloud config set project $PROJECT_ID
!gcloud config set compute/region $REGION

print("Project:")
!gcloud config get-value project
print("\nUser:")
!gcloud config get-value account
print("\nRegion:")
!gcloud config get-value compute/region

## Clone Repository

In [ ]:
%cd /content
!rm -rf {REPO_NAME}
!git clone {REPO_URL}

# Move to repo root
%cd {CLONE_PATH}

# Verify structure
!ls -lh data/raw

## Load Titanic CSV

In [ ]:
# Path to Titanic CSV relative to repo root
DATA_PATH = "data/raw/train.csv"

# Load CSV
df = pd.read_csv(DATA_PATH)

# Quick preview
df.head()

## Dataset Overview

In [ ]:
print("Shape:", df.shape)
print("\nData Types:")
print(df.dtypes)
df.describe(include="all")

## Feature Engineering

### Drop Irrelevant / High-Missing Columns

In [ ]:
columns_to_drop = [
    "PassengerId",
    "Name",
    "Ticket",
    "Cabin",
    "Embarked"
]

df = df.drop(columns=columns_to_drop)
df.head()

## Impute Age



In [ ]:
df["Age"] = df["Age"].fillna(df["Age"].mean())

## Encode Sex

In [ ]:
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})

## Create FamilySize

In [ ]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

## Log Transform Fare

In [ ]:
df["Fare"] = np.log1p(df["Fare"])

## Feature / Target Split


In [ ]:
TARGET = "Survived"

X = df.drop(columns=[TARGET])
y = df[TARGET]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

## Save Processed Dataset

In [ ]:
# Create prefix-aware processed directory
processed_dir = f"data/processed/{DATA_PREP_PREFIX}"
os.makedirs(processed_dir, exist_ok=True)

processed_path = f"{processed_dir}train_processed.csv"

df.to_csv(processed_path, index=False)

print("Saved to:", processed_path)

## Feature Engineering Summary

- Dropped Cabin and Embarked
- Encoded Sex numerically
- Created FamilySize feature
- Imputed Age with mean
- Log-transformed Fare
- Saved processed dataset using DATA_PREP_PREFIX